# MitoFish Dataset Preprocessing for Phylogenetic Embedding

This notebook prepares a curated mitochondrial gene dataset for training a transformer model that learns gene-invariant species embeddings.

The preprocessing pipeline integrates:

- MitoFish mitochondrial gene sequences
- MitoFish taxonomy metadata
- A reference fish phylogenetic tree (The Fish Tree of Life)

The goal is to construct a dataset where each sequence is linked to a **species present in the phylogenetic tree**, allowing later alignment between sequence embeddings and evolutionary relationships.

Main preprocessing steps:

1. Load and type-optimize MitoFish annotation tables
2. Join sequence annotations with taxonomy metadata
3. Parse mitochondrial sequences from FASTA
4. Merge sequences with gene and taxonomy annotations
5. Extract species names from the phylogenetic tree
6. Resolve species name mismatches between the tree and MitoFish
7. Filter sequences to species present in the tree
8. Generate dataset statistics
9. Export the final processed dataset

## Import libraries and define dataset paths

This section imports the required Python libraries and defines the file paths used throughout the preprocessing pipeline.

Key dependencies:

- **pandas** for data manipulation
- **ETE3** for parsing the phylogenetic tree
- **matplotlib** for quick exploratory plots

Dataset directories:

- `datasets/` contains the raw MitoFish tables, FASTA sequences and phylogenetic tree
- `output/` will contain the processed dataset

In [1]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from ete3 import Tree

path_dataset = Path("datasets")
path_out_dir = Path("output")

path_fishtree = path_dataset / "fishtree"
path_fishtree_taxonomy = path_fishtree / "PFC_taxonomy.csv"
path_fishtree_tree = path_fishtree / "actinopt_12k_raxml.tre"

path_mitofish = path_dataset / "MitoFish"
path_mitofish_fasta = path_mitofish / "mitofishdb.fa"
path_mitofish_seq_annotation = path_mitofish / "seq_annotation.parquet"
path_mitofish_seq_description = path_mitofish / "seq_description.parquet"
path_mitofish_seq_heterospecific = path_mitofish / "seq_heterospecific.parquet"
path_mitofish_seq_taxonid = path_mitofish / "seq_taxonid.parquet"
path_mitofish_taxonid_name = path_mitofish / "taxonid_name.parquet"
path_mitofish_taxonid_speciesid = path_mitofish / "taxonid_speciesid.parquet"
path_mitofish_speciesid_lineageid = path_mitofish / "speciesid_lineageid.parquet"

## Load MitoFish sequence annotations

MitoFish provides detailed annotation tables describing the mitochondrial genes present in each sequence record.

This table contains:

- accession identifiers
- gene name
- start and end positions within the mitochondrial genome
- strand orientation
- sequence length

To reduce memory usage, several columns are cast to more efficient data types:

- `gene` → categorical
- `start`, `end`, `length` → 32-bit integers
- `accession` → string

In [2]:
df_mf_seq_an = pd.read_parquet(path_mitofish_seq_annotation).astype({
    'accession': 'string',
    'gene': 'category',
    'start': 'int32',
    'end': 'int32',
    'strand': 'category',
    'length': 'int32'})
df_mf_seq_an

,accession,gene,start,end,strand,length
0,AB000667,12S_rRNA,2469,3417,1,949
1,AB000667,16S_rRNA,3491,3576,1,86
2,AB000667,Cytb,2,857,1,856
3,AB000673,ND5,194,1008,1,815
4,AB000674,COXIII,1,233,1,233
...,...,...,...,...,...,...
1004508,Z97553,Cytb,1,363,1,363
1004509,Z97554,Cytb,1,363,1,363
1004510,Z97555,Cytb,1,363,1,363
1004511,Z97556,Cytb,1,363,1,363


## Load sequence–taxon mapping

Each sequence accession is associated with a **taxonomic identifier (taxon_id)**.

This table links sequence accessions to taxonomic identifiers and creation metadata.

The taxonomic identifier will later be used to retrieve:

- species identity
- taxonomic lineage

In [3]:
df_mf_seq_taxid = pd.read_parquet(path_mitofish_seq_taxonid).astype({
    'accession': 'string',
    'create_date': 'datetime64[ns]',
    'taxon_id': 'int32'
})
df_mf_seq_taxid

,accession,create_date,taxon_id
0,AB000667,1997-02-05,8255
1,AB000668,1997-02-05,8255
2,AB000669,1997-02-05,8255
3,AB000670,1997-02-05,8255
4,AB000671,1997-02-05,8255
...,...,...,...
947202,Z97553,1999-09-20,27763
947203,Z97554,1999-09-20,27763
947204,Z97555,1999-09-20,27763
947205,Z97556,1999-09-20,27763


## Load taxonomic lineage information

MitoFish provides lineage tables describing the hierarchical taxonomy associated with each taxon identifier.

This includes taxonomic ranks such as:

- order
- family
- genus
- species

These tables allow reconstruction of the full taxonomic lineage for each sequence.

In [4]:
df_mf_taxid_name = pd.read_parquet(path_mitofish_taxonid_name).astype({
    'lineage_id': 'int32',
    'lineage_rank': 'category',
    'lineage_name': 'string'
})
df_mf_taxid_name

,lineage_id,lineage_rank,lineage_name
0,1000278,subspecies,Acheilognathus tabira tohokuensis
1,1000982,species,Steindachneridion melanodermatum
2,1001000,species,Sebastes chrysomelas x Sebastes carnatus
3,100124,genus,Pseudobarbus
4,100125,species,Pseudobarbus burchelli
...,...,...,...
60912,999360,species,Hypselobarbus periyarensis
60913,999361,species,Barbodes carnaticus
60914,999678,genus,Oxygaster
60915,999679,species,Oxygaster anomalura


## Map taxon identifiers to species identifiers

MitoFish separates taxonomic identifiers (`taxon_id`) from **species identifiers (`species_id`)**.

This table links each taxon entry to the corresponding species identifier, allowing aggregation of sequences at the species level.

In [5]:
df_mf_taxid_spid = pd.read_parquet(path_mitofish_taxonid_speciesid).astype({
    'taxon_id': 'int32',
    'species_id': 'int32'
})
df_mf_taxid_spid

,taxon_id,species_id
0,1000982,1000982
1,1001000,1001000
2,100125,100125
3,100126,100126
4,1001636,1001636
...,...,...
55177,999359,999359
55178,999360,999360
55179,999361,999361
55180,999679,999679


## Retrieve full lineage hierarchy for each species

This table contains the ordered taxonomic lineage for each species identifier.

It is used to reconstruct hierarchical taxonomy information such as:

- genus
- family
- order

These fields will later be included in the final dataset.

In [6]:
df_mf_spid_linid = pd.read_parquet(path_mitofish_speciesid_lineageid).astype({
    'species_id': 'int32',
    'rank_order': 'int32',
    'rank_name': 'category',
    'lineage_id': 'int32'
})

df_mf_spid_tax = pd.merge(df_mf_spid_linid, df_mf_taxid_name, on="lineage_id")

target_ranks = ['order', 'family', 'genus', 'species']
df_mf_spid_tax = df_mf_spid_tax[df_mf_spid_tax['rank_name'].isin(target_ranks)]

df_mf_spid_tax_piv = df_mf_spid_tax.pivot(
    index='species_id',
    columns='rank_name',
    values='lineage_name'
).reset_index()

df_mf_spid_tax_piv

rank_name,species_id,species,family,genus,order
0,7748,Lampetra fluviatilis,Petromyzontidae,Lampetra,Petromyzontiformes
1,7750,Lampetra planeri,Petromyzontidae,Lampetra,Petromyzontiformes
2,7752,Lampetra aepyptera,Petromyzontidae,Lampetra,Petromyzontiformes
3,7753,Lethenteron reissneri,Petromyzontidae,Lethenteron,Petromyzontiformes
4,7755,Mordacia mordax,Petromyzontidae,Mordacia,Petromyzontiformes
...,...,...,...,...,...
54392,3683097,Ecsenius sp. T76863-1,Blenniidae,Ecsenius,Blenniiformes
54393,3683777,Elates sp. T77057-1,Platycephalidae,Elates,Perciformes
54394,3684156,Danio rerio x Danio albolineatus,Danionidae,Danio,Cypriniformes
54395,3684157,Danio rerio x Danio kyathit,Danionidae,Danio,Cypriniformes


## Merge sequence annotations with taxonomy metadata

All annotation and taxonomy tables are merged to create a unified dataset containing:

- sequence accession
- gene annotation
- taxonomic identifiers
- species identifiers
- lineage information

This step links each mitochondrial gene annotation to its corresponding species.

In [7]:
df_mf_seq_an_taxid = pd.merge(df_mf_seq_an, df_mf_seq_taxid, on="accession")
df_mf_seq_an_spid = pd.merge(df_mf_seq_an_taxid, df_mf_taxid_spid, on="taxon_id")
df_mf_seq_an_tax = pd.merge(df_mf_seq_an_spid, df_mf_spid_tax_piv, on='species_id')
df_mf_seq_an_tax.drop(columns=['create_date', 'taxon_id', 'species_id', 'start', 'end', 'strand'], inplace=True)

df_mf_seq_an_tax

,accession,gene,length,species,family,genus,order
0,AB000667,12S_rRNA,949,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes
1,AB000667,16S_rRNA,86,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes
2,AB000667,Cytb,856,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes
3,AB000673,ND5,815,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes
4,AB000674,COXIII,233,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes
...,...,...,...,...,...,...,...
1004499,Z97553,Cytb,363,Tanganicodus irsacae,Cichlidae,Tanganicodus,Cichliformes
1004500,Z97554,Cytb,363,Tanganicodus irsacae,Cichlidae,Tanganicodus,Cichliformes
1004501,Z97555,Cytb,363,Tanganicodus irsacae,Cichlidae,Tanganicodus,Cichliformes
1004502,Z97556,Cytb,363,Tanganicodus irsacae,Cichlidae,Tanganicodus,Cichliformes


## Parse mitochondrial sequences from FASTA

The mitochondrial sequences are stored in a FASTA file.

The FASTA file is parsed into a table containing:

- accession identifier
- nucleotide sequence

This allows direct merging with the annotation tables using the accession ID.

In [8]:
df_mf_fasta = pd.read_csv(path_mitofish_fasta, lineterminator='>', header=None)
df_mf_fasta = df_mf_fasta[0].str.split('\n', expand=True, n=1).set_axis(["accession", "sequence"], axis=1)
df_mf_fasta['accession'] = df_mf_fasta['accession'].str.strip()
df_mf_fasta['sequence'] = df_mf_fasta['sequence'].str.upper().str.strip()

df_mf_fasta['accession'] = df_mf_fasta['accession'].str.split(';')
df_mf_fasta = df_mf_fasta.explode('accession')

df_mf_fasta = df_mf_fasta.astype({
    'accession': 'string',
    'sequence': 'string'
})

df_mf_fasta

,accession,sequence
0,PQ550629,GGCACTGCTTTAAGCCTTCTTATTCGAGCAGAACTAAGCCAACCAG...
1,MN473720,ACTAAATCAGCCCTCGTCCATAAACCCATATCACAGGACTGAACAC...
2,GU593119,GTGATGAAACTTCGGATCCCTCCTACTACTCTGTCTCTCAGCACAA...
3,MH715312,TGGGATTAGATACCCCACTATGCTTAGTCGTAAACCTCGGTAGGCC...
4,OM978181,GGTGCCTGAGCCGGAATAGTAGGTACAGCCCTGAGCCTGCTAATTC...
...,...,...
580146,GU210318,CGCCAGACTACGAGCATAGCTTAAAACCCAAAGGACTTGGCGGTGC...
580147,MZ723175,CTAGTATTCGGTGCATGAGCTGGAATAGTTGGCACGGCCTTAAGCT...
580148,KF808193,CCTTTATTTAATCTTTGGTGCATGAGCGGGGATAGTGGGTACTGGT...
580149,KF021391,ATGCCTCAACTTAACCCCGCACCCTGGTTCCTAATTATAGTATTCT...


## Validate nucleotide alphabet

Before merging sequences with annotations, we verify the nucleotide alphabet present in the dataset.

This step ensures that sequences contain only valid nucleotide characters (e.g., A, C, G, T, N).

In [9]:
set().union(*df_mf_fasta['sequence'].unique())

{'A', 'B', 'C', 'D', 'G', 'H', 'K', 'M', 'N', 'R', 'S', 'T', 'V', 'W', 'Y'}

## Combine sequences with gene annotations

The sequence table is merged with the annotated gene dataset using the accession identifier.

Duplicate entries are removed based on the combination of:

- sequence
- gene
- species

This ensures that identical gene sequences are not counted multiple times.

In [10]:
df_mf_all_genes = pd.merge(df_mf_seq_an_tax, df_mf_fasta, on="accession")
df_mf_all_genes.drop_duplicates(subset=['sequence', 'gene', 'species'], inplace=True)
df_mf_all_genes.value_counts(subset='species').to_frame()

,count
species,
Amblyraja radiata,6715
Gadus morhua,4710
Chaenogobius annularis,3163
Carcharodon carcharias,2703
Carcharhinus leucas,2594
...,...
cf. Selaroides sp. AWFS-F16-1504,1
cf. Pseudorhombus sp. AWFS-F16-1497,1
cf. Pomadasys sp. AWFS-F16-1492,1


## Load reference phylogenetic tree

The fish phylogeny is loaded using the ETE3 library.

Leaf nodes of the tree correspond to species names.

Species names are normalized by replacing underscores with spaces to match the formatting used in the MitoFish dataset.

In [11]:
t = Tree(str(path_fishtree_tree))
t_leaves = t.get_leaves()
df_leaves = pd.DataFrame([l.name.replace("_", " ") for l in t_leaves], columns=['species'], dtype='string')
df_leaves['species'] = df_leaves['species'].str.replace(r'\b(\w+) \1\b', r'\1', regex=True)

df_leaves

,species
0,Gambusia marshi
1,Gambusia panuco
2,Gambusia regani
3,Gambusia aurata
4,Gambusia hurtadoi
...,...
11633,Polypterus congicus
11634,Polypterus retropinnis
11635,Polypterus mokelembembe
11636,Polypterus ornatipinnis


## Identify tree species without sequence data

Not all species present in the phylogenetic tree have mitochondrial gene sequences available in MitoFish.

This step identifies tree species that:

- are not present in the MitoFish dataset
- are not generic placeholder names (e.g., "sp.")

The missing species names are exported for manual inspection.

In [12]:
df_leaves_not_in_mf = df_leaves[(~df_leaves['species'].isin(df_mf_all_genes['species'])) &
                                (~df_leaves['species'].str.endswith(' sp'))]

for i in range(0, df_leaves_not_in_mf.shape[0], 100):
    path_leaves_not_in_mf = path_out_dir / f"leaves_not_in_mf_{i}-{i+100}.txt"
    df_leaves_not_in_mf.iloc[i:i+100].to_csv(path_leaves_not_in_mf, index=False, header=False)

df_leaves_not_in_mf

,species
10,Gambusia zarskei
119,Limia dominicensis
148,Micropoecilia picta
194,Cyprinodon hubbsi
198,Cyprinodon variegatus ovinus
...,...
11527,Anguilla bengalensis labiata
11532,Anguilla bicolor pacifica
11545,Anguilla australis schmidtii
11624,Polypterus polli


## Resolve species name mismatches using MitoFish mapping tool

Many species mismatches arise due to **taxonomic revisions**, such as genus changes.

To resolve these inconsistencies, the MitoFish species name mapping tool was used:

https://mitofish.aori.u-tokyo.ac.jp/species/mapping

Because the tool only accepts **100 species names per query**, the list of unmatched species was split into multiple batches.

The returned mappings were then combined into a single table.

In [13]:
df_map = pd.concat(
    pd.read_csv(path_fishtree / f"table-mapping-species-names-{i}.csv") for i in range (1, 12)
)

df_map = df_map[df_map['Need Manually Check?'] == "Yes"]
df_map = df_map[df_map['Query Name'].str.split().str[-1].str[:3] ==
                                  df_map['Name in NCBI'].str.split().str[-1].str[:3]]
df_map.drop_duplicates('Query Name', inplace=True)

df_map

,Taxon ID,Query Name,Name in NCBI,Need Manually Check?
1,154033.0,Limia dominicensis,Poecilia dominicensis,Yes
2,69234.0,Micropoecilia picta,Poecilia picta,Yes
4,2802244.0,Cyprinodon variegatus ovinus,Cyprinodon ovinus,Yes
6,30734.0,Garmanella pulchra,Jordanella pulchra,Yes
7,722643.0,Adinia xenica,Fundulus xenicus,Yes
...,...,...,...,...
37,2864443.0,Pisodonophis sangjuensis,Ophichthus sangjuensis,Yes
39,267672.0,Gnathophis nystromi ginanago,Gnathophis ginanago,Yes
40,391210.0,Poeciloconger kapala,Ariosoma kapala,Yes
42,458554.0,Bassanago albescens,Pseudoxenomystax albescens,Yes


## Apply species name remapping

The mappings returned by the MitoFish name mapping tool are applied to the tree species list.

This step corrects outdated or synonymized species names.

After remapping, the number of tree species without sequence data decreased from:

**1038 → 171 species**

In [14]:
name_mapping = dict(zip(df_map['Query Name'], df_map['Name in NCBI']))

df_leaves_remap = df_leaves.replace(name_mapping)
df_leaves_remap_not_in_mf = df_leaves[(~df_leaves_remap['species'].isin(df_mf_all_genes['species'])) &
                                (~df_leaves_remap['species'].str.endswith(' sp'))]

df_leaves_remap_not_in_mf

,species
10,Gambusia zarskei
194,Cyprinodon hubbsi
299,Aphanius farsicus
302,Aphanius splendens
436,Hypsolebias flammeus
...,...
11454,Thalassenchelys coheni
11532,Anguilla bicolor pacifica
11545,Anguilla australis schmidtii
11624,Polypterus polli


## Filter sequences to species present in the phylogenetic tree

Only sequences belonging to species present in the phylogenetic tree are retained.

This ensures that every sequence in the dataset corresponds to a species that has a known position in the phylogeny.

This alignment is required for later experiments where sequence embeddings will be compared with evolutionary distances.

## Dataset statistics

Gene-level statistics are computed to characterize the dataset, including:

- number of species represented per gene
- number of sequences per gene
- average sequence length
- number of sequence variants per species

These statistics help assess dataset balance and guide training strategies for the transformer model.

In [15]:
df_mf_all_genes_on_tree = pd.merge(df_mf_all_genes, df_leaves_remap, on='species')

gene_stats = df_mf_all_genes_on_tree.groupby('gene', observed=True).agg(
    species_with_gene=('species', 'nunique'),
    sequence_count=('gene', 'size'),
    average_length=('length', 'mean'),
    std_length=('length', 'std'),
)

counts = df_mf_all_genes_on_tree.groupby(['species', 'gene'], observed=True).size().reset_index(name='copy_count')
copy_stats = counts.groupby('gene', observed=True)['copy_count'].agg(
    avg_copies_per_species='mean',
    std_copies_per_species='std'
)

gene_stats = gene_stats.join(copy_stats)

print(f"Number of species: {df_mf_all_genes_on_tree['species'].nunique()}")
gene_stats

Number of species: 11457


,species_with_gene,sequence_count,average_length,std_length,avg_copies_per_species,std_copies_per_species
gene,,,,,,
12S_rRNA,8010,32909,607.904525,324.349198,4.108489,8.626174
16S_rRNA,8708,39006,882.584346,557.991372,4.479329,9.581883
ATPase6,4739,17885,654.169304,80.530395,3.774003,11.368877
ATPase8,4595,16077,161.301984,23.495666,3.498803,10.651449
COXI,9862,141821,705.548367,273.278887,14.380552,27.315222
COXII,4136,10657,684.673736,44.251449,2.576644,8.159144
COXIII,4224,11156,731.664396,183.096775,2.641098,8.045424
Cytb,9082,109454,921.649871,278.580635,12.051751,36.767791
ND1,4495,13477,952.961935,113.717158,2.998220,10.942765


## Export processed dataset

The final curated dataset is exported as a Parquet file.

The dataset includes:

- nucleotide sequences
- gene annotations
- species identifiers
- taxonomic lineage (order, family, genus, species)

This processed dataset will be used as input for the transformer model training pipeline.

In [16]:
path_out_dir = Path("output")
path_processed_dataset = path_out_dir / "processed_dataset.parquet"

df_final = df_mf_all_genes_on_tree.astype({tax : 'category' for tax in ['order', 'family', 'genus', 'species'] })
df_final.to_parquet(path_processed_dataset)

df_final


,accession,gene,length,species,family,genus,order,sequence
0,AB000667,12S_rRNA,949,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes,CCTCCACATCGGCCGAGGTCTATACTACGGCTCTTTTCTGTATAAA...
1,AB000667,16S_rRNA,86,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes,CCTCCACATCGGCCGAGGTCTATACTACGGCTCTTTTCTGTATAAA...
2,AB000667,Cytb,856,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes,CCTCCACATCGGCCGAGGTCTATACTACGGCTCTTTTCTGTATAAA...
3,AB000673,ND5,815,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes,CATTAGATTGTGATTCTAAAAACAGAGGTTGAAGCCCTCTTGCCCA...
4,AB000674,COXIII,233,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes,CCCTTCACTATTGCAGATGGGGTTTATGGCGCCACATTCTTCGTTG...
...,...,...,...,...,...,...,...,...
472850,Z97552,Cytb,363,Tanganicodus irsacae,Cichlidae,Tanganicodus,Cichliformes,GCAAACGACGCACTGGTTGATCTCCCAGCTCCTTCAAACATTTCCG...
472851,Z97553,Cytb,363,Tanganicodus irsacae,Cichlidae,Tanganicodus,Cichliformes,GCAAACGACGCACTGGTTGATCTCCCAGCTCCTTCAAACATTTCCG...
472852,Z97554,Cytb,363,Tanganicodus irsacae,Cichlidae,Tanganicodus,Cichliformes,GCAAACGACGCACTGGTTGATCTCCCAGCTCCTTCAAACATTTCCG...
472853,Z97555,Cytb,363,Tanganicodus irsacae,Cichlidae,Tanganicodus,Cichliformes,NNNNNNNNNNNNNNNGTTGATCTCCCAGCTCCTTCAAACATTTCCG...
